# 🤖 InvestAI — Módulo de Clasificación SVC (BUY/SELL)

**Proyecto:** Sistema Ernesto Investing AI  
**Módulo:** Clasificador SVC con optimización GridSearchCV  
**Fuente de datos:** MongoDB Atlas (`precios_ohlcv`)  
**Salida:** MongoDB Atlas (`predicciones`, `metricas_modelos`)  

---

Este notebook continúa el pipeline del módulo de Ingesta de Datos:
lee los precios e indicadores técnicos ya almacenados en MongoDB,
construye un conjunto de *features* y una variable objetivo binaria
(BUY/SELL), entrena un **Support Vector Classifier** optimizado con
**GridSearchCV** y persiste de vuelta en MongoDB tanto la señal
vigente de cada ticker como las métricas de desempeño del modelo.

**Tickers procesados:** FSM, VOLCABC1.LM, ABX.TO, BVN, BHP

> ⚠️ **Requisito previo:** Haber ejecutado el notebook de Ingesta de
> Datos (o tener ya datos en `precios_ohlcv`) y tener `MONGO_URI`
> configurado en los Secrets de Colab.

## Sección 1 — Instalación e Importación de Librerías

In [6]:
# pymongo[srv]: cliente MongoDB con soporte de conexión Atlas (SRV)
# scikit-learn: SVC, GridSearchCV, métricas y preprocesamiento
!pip install "pymongo[srv]" scikit-learn --quiet

print("✅ Dependencias instaladas correctamente.")

✅ Dependencias instaladas correctamente.


In [7]:
import warnings
warnings.filterwarnings('ignore')

import math
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix
)

from pymongo import MongoClient, UpdateOne
from pymongo.errors import ConnectionFailure, BulkWriteError

from google.colab import userdata

print("✅ Librerías importadas correctamente.")
print(f"   numpy   {np.__version__}")
print(f"   pandas  {pd.__version__}")

✅ Librerías importadas correctamente.
   numpy   2.0.2
   pandas  2.2.2


## Sección 2 — Configuración Global

In [8]:
# ─── Tickers del estudio ───────────────────────────────────────────────────
TICKERS = ['FSM', 'VOLCABC1.LM', 'ABX.TO', 'BVN', 'BHP']

EMPRESAS_META = {
    'FSM':         {'nombre': 'Fortuna Silver Mines Inc.',              'moneda': 'USD'},
    'VOLCABC1.LM': {'nombre': 'Volcan Compañía Minera S.A.A.',          'moneda': 'PEN'},
    'ABX.TO':      {'nombre': 'Barrick Gold Corporation',               'moneda': 'CAD'},
    'BVN':         {'nombre': 'Compañía de Minas Buenaventura S.A.A.',  'moneda': 'USD'},
    'BHP':         {'nombre': 'BHP Group Limited',                      'moneda': 'USD'},
}

# ─── Parámetros del modelo ─────────────────────────────────────────────────
RATIO_TRAIN     = 0.80     # 80% entrenamiento / 20% prueba (partición TEMPORAL, sin shuffle)
CV_FOLDS        = 5        # Folds para TimeSeriesSplit dentro del GridSearchCV
MIN_REGISTROS   = 80       # Mínimo de registros requeridos para entrenar con confiabilidad

# Grilla de hiperparámetros a explorar (según especificación: kernels linear/rbf, C, gamma)
PARAM_GRID = {
    'svc__kernel': ['linear', 'rbf'],
    'svc__C':      [0.1, 1, 10, 100],
    'svc__gamma':  ['scale', 'auto'],
}

# Columnas de indicadores ya calculadas por el módulo de Ingesta (en precios_ohlcv)
COLUMNAS_INDICADORES = ['sma_20', 'sma_50', 'ema_12', 'ema_26', 'rsi_14']

# ─── Configuración de MongoDB ──────────────────────────────────────────────
DB_NOMBRE        = 'investai'
COL_PRECIOS      = 'precios_ohlcv'      # Colección de entrada (lectura)
COL_PREDICCIONES = 'predicciones'       # Colección de salida: señal vigente por ticker
COL_METRICAS     = 'metricas_modelos'   # Colección de salida: métricas de desempeño

print("✅ Configuración cargada.")
print(f"   Tickers      : {TICKERS}")
print(f"   Train/Test   : {int(RATIO_TRAIN*100)}% / {int((1-RATIO_TRAIN)*100)}%")
print(f"   CV folds     : {CV_FOLDS} (TimeSeriesSplit)")
print(f"   Grid de C    : {PARAM_GRID['svc__C']}")
print(f"   Grid kernels : {PARAM_GRID['svc__kernel']}")

✅ Configuración cargada.
   Tickers      : ['FSM', 'VOLCABC1.LM', 'ABX.TO', 'BVN', 'BHP']
   Train/Test   : 80% / 19%
   CV folds     : 5 (TimeSeriesSplit)
   Grid de C    : [0.1, 1, 10, 100]
   Grid kernels : ['linear', 'rbf']


## Sección 3 — Conexión a MongoDB Atlas

Reutiliza el mismo patrón de conexión segura del módulo de Ingesta:
la URI se lee desde los **Secrets de Colab** (`MONGO_URI`).

In [9]:
try:
    MONGO_URI = userdata.get('MONGO_URI')
    if not MONGO_URI:
        raise ValueError("El secret MONGO_URI está vacío.")
except Exception as e:
    raise RuntimeError(
        f"No se encontró el secret MONGO_URI: {e}\n"
        "Agrega tu URI en: Panel izquierdo → 🔑 → Agregar nuevo secreto → MONGO_URI"
    )

try:
    cliente = MongoClient(
        MONGO_URI,
        serverSelectionTimeoutMS=10_000,
        connectTimeoutMS=10_000,
    )
    cliente.admin.command('ping')
    print("✅ Conexión a MongoDB Atlas establecida correctamente.")
except ConnectionFailure as e:
    raise RuntimeError(
        f"No se pudo conectar a MongoDB Atlas: {e}\n"
        "Verifica que la URI sea correcta y que la IP de Colab esté en la lista blanca de Atlas."
    )

db                = cliente[DB_NOMBRE]
col_precios       = db[COL_PRECIOS]
col_predicciones  = db[COL_PREDICCIONES]
col_metricas      = db[COL_METRICAS]

# Índices únicos para mantener idempotencia (upsert sin duplicados)
col_predicciones.create_index([('ticker', 1)], unique=True, name='idx_ticker_unico')
col_metricas.create_index(
    [('ticker', 1), ('modelo', 1), ('entrenado_en', -1)],
    name='idx_ticker_modelo_fecha'
)

n_precios = col_precios.count_documents({})
print(f"   Base de datos       : {DB_NOMBRE}")
print(f"   Colección lectura   : {COL_PRECIOS}  ({n_precios:,} documentos disponibles)")
print(f"   Colección salida 1  : {COL_PREDICCIONES}")
print(f"   Colección salida 2  : {COL_METRICAS}")

if n_precios == 0:
    print("\n⚠️  ADVERTENCIA: la colección 'precios_ohlcv' está vacía.")
    print("    Ejecuta primero el notebook de Ingesta de Datos.")

✅ Conexión a MongoDB Atlas establecida correctamente.
   Base de datos       : investai
   Colección lectura   : precios_ohlcv  (1,252 documentos disponibles)
   Colección salida 1  : predicciones
   Colección salida 2  : metricas_modelos


In [10]:
import requests

# Obtener la IP pública actual de la sesión de Colab
response = requests.get('https://api.ipify.org?format=json')
public_ip = response.json()['ip']
print(f"Tu dirección IP pública actual de Colab es: {public_ip}")

Tu dirección IP pública actual de Colab es: 136.113.173.75


## Sección 4 — Funciones de Utilidad

Helpers para serialización JSON-segura, lectura desde MongoDB y
construcción de *features* / *target* sin fuga de información futura.

In [11]:
def nan_a_none(valor):
    """
    Convierte NaN / Inf a None para que pymongo los serialice
    como null (JSON válido) en lugar de fallar.

    Parámetros:
        valor : cualquier tipo numérico o None

    Retorna:
        float redondeado a 6 decimales, o None si es NaN/Inf/None
    """
    if valor is None:
        return None
    try:
        if math.isnan(valor) or math.isinf(valor):
            return None
        return round(float(valor), 6)
    except (TypeError, ValueError):
        return None

In [12]:
def leer_precios_mongodb(ticker: str) -> pd.DataFrame | None:
    """
    Lee de MongoDB todos los documentos OHLCV + indicadores de un
    ticker, ordenados cronológicamente, y los convierte en DataFrame.

    Parámetros:
        ticker : símbolo bursátil (ej. 'BVN')

    Retorna:
        pd.DataFrame indexado por fecha, o None si no hay datos
    """
    cursor = col_precios.find(
        {'ticker': ticker},
        projection={'_id': 0}
    ).sort('fecha', 1)

    registros = list(cursor)
    if not registros:
        return None

    df = pd.DataFrame(registros)
    df['fecha'] = pd.to_datetime(df['fecha'])
    df = df.set_index('fecha').sort_index()

    # Eliminar duplicados de fecha (por seguridad, aunque el índice único ya lo evita)
    df = df[~df.index.duplicated(keep='last')]

    return df

In [13]:
def construir_features_target(df: pd.DataFrame) -> pd.DataFrame:
    """
    Construye el conjunto de features (indicadores técnicos ya
    calculados en la ingesta) y la variable objetivo binaria 'target'.

    target = 1 (BUY)  si el cierre de MAÑANA > cierre de HOY
    target = 0 (SELL) en caso contrario

    ⚠️ PREVENCIÓN DE DATA LEAKAGE:
    El target se construye con shift(-1) — es decir, mira el precio
    del día SIGUIENTE, nunca usa información que no estaría disponible
    en el momento de la predicción. Las features (SMA, EMA, RSI) ya
    fueron calculadas en el módulo de Ingesta usando solo datos
    pasados hasta cada fecha (rolling/ewm), por lo que tampoco filtran
    información futura.

    La última fila del DataFrame queda sin target válido (no se conoce
    el cierre de "mañana" todavía) — se descarta del entrenamiento pero
    se conserva aparte para generar la PREDICCIÓN VIGENTE.

    Parámetros:
        df : DataFrame con OHLCV + indicadores (desde MongoDB)

    Retorna:
        DataFrame con columnas de features + 'target', sin NaN
    """
    d = df.copy()

    # Variable objetivo: ¿el precio sube mañana? (shift hacia adelante = mirar futuro próximo)
    d['cierre_manana'] = d['cierre'].shift(-1)
    d['target'] = (d['cierre_manana'] > d['cierre']).astype(int)

    # Features adicionales derivadas (calculadas solo con datos pasados/actuales)
    d['retorno_1d']      = d['cierre'].pct_change(1)
    d['rango_intradia']  = (d['maximo'] - d['minimo']) / d['cierre']
    d['precio_sobre_sma20'] = d['cierre'] / d['sma_20'] - 1
    d['precio_sobre_sma50'] = d['cierre'] / d['sma_50'] - 1
    d['cruce_ema']        = d['ema_12'] / d['ema_26'] - 1

    return d


print("✅ Funciones de utilidad definidas:")
print("   · nan_a_none(valor)")
print("   · leer_precios_mongodb(ticker)")
print("   · construir_features_target(df)")

✅ Funciones de utilidad definidas:
   · nan_a_none(valor)
   · leer_precios_mongodb(ticker)
   · construir_features_target(df)


## Sección 5 — Entrenamiento del Clasificador SVC

Pipeline completo: `StandardScaler → SVC` optimizado con `GridSearchCV`
usando `TimeSeriesSplit` (validación cruzada respetando el orden
cronológico, sin mezclar pasado y futuro).

In [14]:
def entrenar_svc_ticker(ticker: str) -> dict | None:
    """
    Pipeline completo de entrenamiento del SVC para un ticker:
        1. Lee precios + indicadores desde MongoDB.
        2. Construye features y target (sin fuga de información futura).
        3. Partición TEMPORAL 80/20 (sin shuffle, respeta cronología).
        4. Normaliza con StandardScaler (fit SOLO en train).
        5. Entrena SVC con GridSearchCV + TimeSeriesSplit.
        6. Evalúa en el conjunto de prueba (accuracy/precision/recall/F1).
        7. Genera la predicción de la señal VIGENTE (último día disponible).

    Parámetros:
        ticker : símbolo bursátil (ej. 'BVN')

    Retorna:
        dict con resultados (métricas, mejores hiperparámetros, señal
        vigente), o None si no hay datos suficientes.
    """
    # ── Paso 1: Lectura desde MongoDB ───────────────────────────────────────
    df_raw = leer_precios_mongodb(ticker)
    if df_raw is None or len(df_raw) < MIN_REGISTROS:
        n = 0 if df_raw is None else len(df_raw)
        print(f"   ⚠️  {ticker}: solo {n} registros en MongoDB — se requieren ≥{MIN_REGISTROS}.")
        return None

    # ── Paso 2: Features + target ───────────────────────────────────────────
    df_feat = construir_features_target(df_raw)

    columnas_features = COLUMNAS_INDICADORES + [
        'retorno_1d', 'rango_intradia',
        'precio_sobre_sma20', 'precio_sobre_sma50', 'cruce_ema',
    ]

    # Fila más reciente: features completas pero target desconocido (es "mañana")
    fila_vigente = df_feat.iloc[[-1]].copy()

    # Conjunto de entrenamiento/prueba: descarta filas con NaN en features o target
    # (NaN en features ocurre durante el período de calentamiento de SMA-50, etc.
    #  NaN en target ocurre solo en la última fila, ya excluida arriba)
    df_modelo = df_feat.dropna(subset=columnas_features + ['target']).copy()

    if len(df_modelo) < MIN_REGISTROS:
        print(f"   ⚠️  {ticker}: solo {len(df_modelo)} registros válidos tras dropna — insuficiente.")
        return None

    # ── Paso 3: Partición TEMPORAL (sin shuffle) ────────────────────────────
    n          = len(df_modelo)
    corte      = int(n * RATIO_TRAIN)
    df_train   = df_modelo.iloc[:corte]
    df_test    = df_modelo.iloc[corte:]

    if len(df_test) < 5:
        print(f"   ⚠️  {ticker}: conjunto de prueba demasiado pequeño ({len(df_test)} filas).")
        return None

    X_train, y_train = df_train[columnas_features].values, df_train['target'].values
    X_test,  y_test  = df_test[columnas_features].values,  df_test['target'].values
    X_vigente        = fila_vigente[columnas_features].values

    # ── Paso 4 + 5: Pipeline StandardScaler + SVC, optimizado con GridSearchCV ─
    # El scaler se ajusta (fit) ÚNICAMENTE con datos de entrenamiento dentro
    # del pipeline; sklearn garantiza que no haya fuga hacia validación/test.
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('svc', SVC(probability=True, random_state=42, cache_size=500)),
    ])

    # TimeSeriesSplit: cada fold de validación usa solo datos POSTERIORES
    # al fold de entrenamiento correspondiente — respeta la cronología.
    n_splits_seguro = min(CV_FOLDS, max(2, len(X_train) // 20))
    tscv = TimeSeriesSplit(n_splits=n_splits_seguro)

    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=PARAM_GRID,
        cv=tscv,
        scoring='f1_macro',
        n_jobs=-1,
        refit=True,
    )

    grid_search.fit(X_train, y_train)
    mejor_modelo  = grid_search.best_estimator_
    mejores_pars  = grid_search.best_params_
    mejor_cv_f1   = grid_search.best_score_

    # ── Paso 6: Evaluación en el conjunto de prueba (held-out, nunca visto) ──
    y_pred = mejor_modelo.predict(X_test)

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec  = recall_score(y_test, y_pred, zero_division=0)
    f1   = f1_score(y_test, y_pred, zero_division=0)
    cm   = confusion_matrix(y_test, y_pred, labels=[0, 1])

    # ── Paso 7: Predicción de la señal VIGENTE (último día disponible) ──────
    prob_vigente = mejor_modelo.predict_proba(X_vigente)[0]
    pred_vigente = mejor_modelo.predict(X_vigente)[0]
    señal        = 'BUY' if pred_vigente == 1 else 'SELL'
    confianza    = float(prob_vigente[1]) if pred_vigente == 1 else float(prob_vigente[0])

    return {
        'ticker': ticker,
        'n_train': len(df_train),
        'n_test':  len(df_test),
        'fecha_corte': df_train.index[-1].strftime('%Y-%m-%d'),
        'fecha_vigente': fila_vigente.index[0].strftime('%Y-%m-%d'),
        'mejores_hiperparametros': mejores_pars,
        'cv_f1_macro': nan_a_none(mejor_cv_f1),
        'metricas': {
            'accuracy':  nan_a_none(acc),
            'precision': nan_a_none(prec),
            'recall':    nan_a_none(rec),
            'f1':        nan_a_none(f1),
        },
        'matriz_confusion': cm.tolist(),  # [[TN, FP], [FN, TP]]
        'señal_vigente': señal,
        'confianza_vigente': nan_a_none(confianza),
        'ultimo_cierre': nan_a_none(fila_vigente['cierre'].iloc[0]),
        'columnas_features': columnas_features,
    }


print("✅ Función entrenar_svc_ticker() definida.")

✅ Función entrenar_svc_ticker() definida.


## Sección 6 — Funciones de Persistencia en MongoDB

In [15]:
def guardar_prediccion(resultado: dict) -> None:
    """
    Guarda (upsert) la señal vigente del ticker en la colección
    'predicciones'. Un documento por ticker — siempre refleja
    la última predicción generada.

    Parámetros:
        resultado : dict devuelto por entrenar_svc_ticker()
    """
    meta = EMPRESAS_META.get(resultado['ticker'], {})
    doc = {
        'ticker'         : resultado['ticker'],
        'nombre'         : meta.get('nombre', resultado['ticker']),
        'modelo'         : 'SVC',
        'señal'          : resultado['señal_vigente'],
        'confianza'      : resultado['confianza_vigente'],
        'ultimo_cierre'  : resultado['ultimo_cierre'],
        'fecha_referencia': resultado['fecha_vigente'],
        'hiperparametros': resultado['mejores_hiperparametros'],
        'actualizado_en' : datetime.now(timezone.utc),
    }

    col_predicciones.update_one(
        filter={'ticker': resultado['ticker']},
        update={'$set': doc},
        upsert=True
    )


def guardar_metricas(resultado: dict) -> None:
    """
    Inserta un nuevo registro histórico de métricas en la colección
    'metricas_modelos'. A diferencia de 'predicciones', aquí se
    acumula un historial (insert, no upsert) para poder trazar la
    evolución del desempeño del modelo a lo largo del tiempo.

    Parámetros:
        resultado : dict devuelto por entrenar_svc_ticker()
    """
    meta = EMPRESAS_META.get(resultado['ticker'], {})
    doc = {
        'ticker'         : resultado['ticker'],
        'nombre'         : meta.get('nombre', resultado['ticker']),
        'modelo'         : 'SVC',
        'n_train'        : resultado['n_train'],
        'n_test'         : resultado['n_test'],
        'fecha_corte_train_test': resultado['fecha_corte'],
        'hiperparametros': resultado['mejores_hiperparametros'],
        'cv_f1_macro'    : resultado['cv_f1_macro'],
        'metricas'       : resultado['metricas'],
        'matriz_confusion': resultado['matriz_confusion'],
        'columnas_features': resultado['columnas_features'],
        'entrenado_en'   : datetime.now(timezone.utc),
    }

    col_metricas.insert_one(doc)


print("✅ Funciones de persistencia definidas:")
print("   · guardar_prediccion(resultado)  → upsert en 'predicciones'")
print("   · guardar_metricas(resultado)    → insert histórico en 'metricas_modelos'")

✅ Funciones de persistencia definidas:
   · guardar_prediccion(resultado)  → upsert en 'predicciones'
   · guardar_metricas(resultado)    → insert histórico en 'metricas_modelos'


## Sección 7 — Pipeline Principal: Entrenar y Guardar para los 5 Tickers

In [16]:
resumen_resultados = []

print("=" * 70)
print("  ENTRENAMIENTO SVC — InvestAI  (GridSearchCV + TimeSeriesSplit)")
print("=" * 70)

for ticker in TICKERS:
    meta = EMPRESAS_META.get(ticker, {})
    print(f"\n🤖 {ticker}  ({meta.get('nombre', '')})")

    resultado = entrenar_svc_ticker(ticker)

    if resultado is None:
        resumen_resultados.append({
            'ticker': ticker, 'estado': '❌ Sin datos suficientes',
            'señal': '—', 'accuracy': None, 'f1': None,
        })
        continue

    m = resultado['metricas']
    print(f"   Train/Test          : {resultado['n_train']} / {resultado['n_test']} registros")
    print(f"   Corte temporal       : hasta {resultado['fecha_corte']}")
    print(f"   Mejores hiperparams  : {resultado['mejores_hiperparametros']}")
    print(f"   CV F1-macro          : {resultado['cv_f1_macro']}")
    print(f"   Accuracy (test)      : {m['accuracy']}")
    print(f"   Precision (test)     : {m['precision']}")
    print(f"   Recall (test)        : {m['recall']}")
    print(f"   F1-score (test)      : {m['f1']}")
    print(f"   Matriz confusión     : {resultado['matriz_confusion']}  (formato [[TN,FP],[FN,TP]])")
    print(f"   📌 Señal vigente     : {resultado['señal_vigente']}  "
          f"(confianza: {resultado['confianza_vigente']:.2%})"
          if resultado['confianza_vigente'] is not None else
          f"   📌 Señal vigente     : {resultado['señal_vigente']}")

    # ── Guardar en MongoDB ───────────────────────────────────────────────────
    guardar_prediccion(resultado)
    guardar_metricas(resultado)
    print(f"   ✅ Guardado en MongoDB → 'predicciones' (upsert) y 'metricas_modelos' (insert)")

    resumen_resultados.append({
        'ticker': ticker,
        'estado': '✅ OK',
        'señal': resultado['señal_vigente'],
        'accuracy': m['accuracy'],
        'f1': m['f1'],
    })

print("\n" + "=" * 70)
print("  RESUMEN FINAL")
print("=" * 70)
df_resumen = pd.DataFrame(resumen_resultados)
print(df_resumen.to_string(index=False))
print("=" * 70)

  ENTRENAMIENTO SVC — InvestAI  (GridSearchCV + TimeSeriesSplit)

🤖 FSM  (Fortuna Silver Mines Inc.)
   Train/Test          : 161 / 41 registros
   Corte temporal       : hasta 2026-05-04
   Mejores hiperparams  : {'svc__C': 0.1, 'svc__gamma': 'scale', 'svc__kernel': 'linear'}
   CV F1-macro          : 0.462509
   Accuracy (test)      : 0.512195
   Precision (test)     : 0.454545
   Recall (test)        : 0.555556
   F1-score (test)      : 0.5
   Matriz confusión     : [[11, 12], [8, 10]]  (formato [[TN,FP],[FN,TP]])
   📌 Señal vigente     : BUY  (confianza: 54.32%)
   ✅ Guardado en MongoDB → 'predicciones' (upsert) y 'metricas_modelos' (insert)

🤖 VOLCABC1.LM  (Volcan Compañía Minera S.A.A.)
   Train/Test          : 158 / 40 registros
   Corte temporal       : hasta 2026-05-04
   Mejores hiperparams  : {'svc__C': 10, 'svc__gamma': 'scale', 'svc__kernel': 'rbf'}
   CV F1-macro          : 0.430142
   Accuracy (test)      : 0.55
   Precision (test)     : 0.56
   Recall (test)        : 0.

## Sección 8 — Verificación Final

Consulta directa a MongoDB para confirmar que las predicciones y
métricas quedaron correctamente almacenadas.

In [17]:
print("=" * 70)
print("  VERIFICACIÓN — Consulta a MongoDB Atlas")
print("=" * 70)

# ── 1. Predicciones vigentes (una por ticker) ──────────────────────────────
print(f"\n📌 Colección '{COL_PREDICCIONES}' — señales vigentes:")
n_predicciones = col_predicciones.count_documents({})
print(f"   Total de documentos: {n_predicciones}  (esperado: {len(TICKERS)}, uno por ticker)")

for doc in col_predicciones.find({}, {'_id': 0}).sort('ticker', 1):
    conf_str = f"{doc['confianza']:.2%}" if doc.get('confianza') is not None else 'N/A'
    print(f"   {doc['ticker']:<14}  señal: {doc['señal']:<5}  "
          f"confianza: {conf_str}  cierre: {doc['ultimo_cierre']}")

# ── 2. Métricas históricas (más reciente por ticker) ────────────────────────
print(f"\n📊 Colección '{COL_METRICAS}' — métricas más recientes por ticker:")
n_metricas_total = col_metricas.count_documents({})
print(f"   Total de documentos históricos: {n_metricas_total}")

pipeline_ultimas_metricas = [
    {'$sort': {'entrenado_en': -1}},
    {'$group': {
        '_id': '$ticker',
        'accuracy': {'$first': '$metricas.accuracy'},
        'precision': {'$first': '$metricas.precision'},
        'recall': {'$first': '$metricas.recall'},
        'f1': {'$first': '$metricas.f1'},
        'hiperparametros': {'$first': '$hiperparametros'},
    }},
    {'$sort': {'_id': 1}}
]
for doc in col_metricas.aggregate(pipeline_ultimas_metricas):
    print(f"   {doc['_id']:<14}  "
          f"acc: {doc['accuracy']:.4f}  "
          f"prec: {doc['precision']:.4f}  "
          f"rec: {doc['recall']:.4f}  "
          f"f1: {doc['f1']:.4f}  "
          f"params: {doc['hiperparametros']}")

# ── 3. Ejemplo de documento completo ────────────────────────────────────────
print("\n📄 Ejemplo de documento completo en 'metricas_modelos' (BVN, más reciente):")
doc_ejemplo = col_metricas.find_one(
    {'ticker': 'BVN'},
    sort=[('entrenado_en', -1)],
    projection={'_id': 0}
)
if doc_ejemplo:
    for clave, valor in doc_ejemplo.items():
        if isinstance(valor, datetime):
            valor = valor.strftime('%Y-%m-%d %H:%M:%S UTC')
        print(f"   {clave:<25}: {valor}")
else:
    print("   ⚠️  No se encontró documento de ejemplo para BVN.")

# ── 4. Sanity checks ─────────────────────────────────────────────────────────
señales_invalidas = col_predicciones.count_documents(
    {'señal': {'$nin': ['BUY', 'SELL']}}
)
print(f"\n✅ Predicciones con señal inválida (≠ BUY/SELL): {señales_invalidas}  (debe ser 0)")

print("\n" + "=" * 70)
print("  ✅ Entrenamiento, predicción y verificación completados correctamente.")
print("=" * 70)

  VERIFICACIÓN — Consulta a MongoDB Atlas

📌 Colección 'predicciones' — señales vigentes:
   Total de documentos: 5  (esperado: 5, uno por ticker)
   ABX.TO          señal: SELL   confianza: 42.69%  cierre: 55.560001
   BHP             señal: BUY    confianza: 55.90%  cierre: 83.330002
   BVN             señal: BUY    confianza: 56.21%  cierre: 29.719999
   FSM             señal: BUY    confianza: 54.32%  cierre: 8.72
   VOLCABC1.LM     señal: SELL   confianza: 58.99%  cierre: 0.861

📊 Colección 'metricas_modelos' — métricas más recientes por ticker:
   Total de documentos históricos: 5
   ABX.TO          acc: 0.4390  prec: 0.4211  rec: 0.4000  f1: 0.4103  params: {'svc__C': 1, 'svc__gamma': 'scale', 'svc__kernel': 'rbf'}
   BHP             acc: 0.4634  prec: 0.5000  rec: 0.3182  f1: 0.3889  params: {'svc__C': 10, 'svc__gamma': 'scale', 'svc__kernel': 'rbf'}
   BVN             acc: 0.5366  prec: 0.4783  rec: 0.6111  f1: 0.5366  params: {'svc__C': 100, 'svc__gamma': 'scale', 'svc__kerne

## Sección 9 — Cierre de Conexión

In [18]:
cliente.close()
print("✅ Conexión a MongoDB Atlas cerrada.")
print()
print("📌 Para reconectar en otra sesión, vuelve a ejecutar desde la Sección 3.")

✅ Conexión a MongoDB Atlas cerrada.

📌 Para reconectar en otra sesión, vuelve a ejecutar desde la Sección 3.
